# Elaborative Rehearsal (B) — Training (stage 1)

Trains `t5-small` on `06`'s pairs. Same structure as `05`; inverted manipulation check.

| | A (`05`) | B (this) |
|---|---|---|
| target | sentences lifted from input | rewritten summary |
| novel n-gram ratio | low, seams only | clearly **above** that |
| verbatim snapping | yes | **no** |

Manipulation check is the point: if B's output is as copy-heavy as A's, the A/B distinction collapses. Measured against the **source**, not the label.

Prior work: RECOMP (ICLR 2024) §3.2 — supervised half of teacher-distillation recipe, XSum standing in for the teacher.


In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()
API_KEY_SET = bool(__import__("os").environ.get("NVIDIA_NIM_API_KEY"))

import datasets
import evaluate
import numpy as np
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
)

from src.pipeline.rehearsal import novel_ngram_ratio

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load data/model

`val` drives checkpoint selection; `test` untouched until §5.


In [2]:
MODEL_NAME = "google/flan-t5-base"  # swapped 2026-08-08 from t5-small (60M) -- flan-t5-base (250M) is instruction-tuned, so it should follow "shorten, don't summarize" better even before fine-tuning, not just be bigger. Same T5 tokenizer family -- 06's cached data needs no changes.
DATA_DIR = Path("data/processed/rehearsal_elaborative")
OUTPUT_DIR = "experiments/rehearsal_elaborative_flant5base"  # separate from t5-small's checkpoint -- the existing stage-2 models (both self_conditioned_ratio settings) still read the old one, untouched
TRAIN_SIZE = 300
VAL_SIZE = 300
TEST_SIZE = 300

DATA_READY = all((DATA_DIR / split).exists() for split in ("train", "val", "test"))
if not DATA_READY:
    print(f"No tokenized data at {DATA_DIR} — run 06_rehearsal_elaborative_prep.ipynb first.")
else:
    full_train = datasets.Dataset.load_from_disk(str(DATA_DIR / "train"))
    full_val = datasets.Dataset.load_from_disk(str(DATA_DIR / "val"))
    full_test = datasets.Dataset.load_from_disk(str(DATA_DIR / "test"))

    train_dataset = full_train.select(range(min(TRAIN_SIZE, len(full_train))))
    val_dataset = full_val.select(range(min(VAL_SIZE, len(full_val))))
    test_dataset = full_test.select(range(min(TEST_SIZE, len(full_test))))
    print(f"Using {len(train_dataset)}/{len(full_train)} train, "
          f"{len(val_dataset)}/{len(full_val)} val, {len(test_dataset)}/{len(full_test)} test rows")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    print(f"{MODEL_NAME} loaded, parameters: {sum(p.numel() for p in model.parameters()):,}")

Using 300/3000 train, 299/299 val, 300/300 test rows
google/flan-t5-base loaded, parameters: 247,577,856


## 2. Metrics

ROUGE-L via `compute_metrics`; novel n-gram ratio in a separate callback (compares against source, not label).


In [3]:
rouge = evaluate.load("rouge")


def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    return {"rougeL": result["rougeL"]}


def average_novel_ngram_ratio(model, tokenizer, eval_dataset, n: int = 3,
                              sample_size: int = 10, max_new_tokens: int = 64) -> float:
    """Generates from a slice of eval_dataset and returns the average novel
    n-gram ratio against the *source* (input), not the label. Shared by the
    callback and the final test evaluation so both use identical logic."""
    examples = eval_dataset.select(range(min(sample_size, len(eval_dataset))))
    was_training = model.training
    model.eval()
    device = next(model.parameters()).device

    ratios = []
    for example in examples:
        input_ids = torch.tensor([example["input_ids"]]).to(device)
        with torch.no_grad():
            output_ids = model.generate(input_ids=input_ids, max_new_tokens=max_new_tokens)
        generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        source = tokenizer.decode(example["input_ids"], skip_special_tokens=True)
        ratios.append(novel_ngram_ratio(generated, source, n=n))

    if was_training:
        model.train()
    return sum(ratios) / len(ratios) if ratios else 0.0


class NovelNGramCallback(TrainerCallback):
    """Logs B's novel n-gram ratio on a validation slice at every eval.

    For B this should stay clearly above zero — it is the evidence that
    elaborative rehearsal is doing something A does not. A ratio drifting
    toward zero means B has collapsed into extraction and the A/B contrast
    has quietly disappeared.
    """

    def __init__(self, tokenizer, eval_dataset, n: int = 3, sample_size: int = 10, max_new_tokens: int = 64):
        self.tokenizer = tokenizer
        self.eval_dataset = eval_dataset
        self.n = n
        self.sample_size = sample_size
        self.max_new_tokens = max_new_tokens

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        ratio = average_novel_ngram_ratio(
            model, self.tokenizer, self.eval_dataset, n=self.n,
            sample_size=self.sample_size, max_new_tokens=self.max_new_tokens,
        )
        print(f"  [manipulation check] novel {self.n}-gram ratio vs source: {ratio:.4f} (higher is better for B)")

## 3. Train

`predict_with_generate=True` (ROUGE-L needs text); `generation_max_length=64` matches XSum's one-sentence targets.


### MPS speed test (2026-08-09) — try CPU instead

Diagnosing the MPS slowdown; superseded by the generate-in-eval-loop fix below.


In [ ]:
model.to("cpu")
print("model is now on:", next(model.parameters()).device)


In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# predict_with_generate + NovelNGramCallback both trigger autoregressive
# generation every eval -- fine on t5-small, but generate() on MPS is a known
# slow path (82h ETA measured for flan-t5-base with both enabled, 2026-08-09).
# Loss-based eval (no generation) is a plain forward pass and doesn't hit
# that path. The real generation-quality check (ROUGE + novel n-gram) still
# runs once at the end against the held-out test set (§5 below), so nothing
# is lost -- it's just not repeated 3x during training for no benefit.
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    predict_with_generate=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

trainer.train()

  0%|          | 0/114 [00:00<?, ?it/s]/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
  4%|▎         | 4/114 [08:22<5:34:34, 182.50s/it]

## 4. Save final model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to: {OUTPUT_DIR}")

## 5. Final evaluation — held-out test set

Never touched during training — this is the reported number.


In [ ]:
if not DATA_READY:
    print("No data — skipping final test evaluation.")
else:
    test_trainer = Seq2SeqTrainer(
        model=model,
        args=Seq2SeqTrainingArguments(
            output_dir=OUTPUT_DIR,
            per_device_eval_batch_size=8,
            predict_with_generate=True,
            generation_max_length=64,
            report_to="none",
        ),
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    test_metrics = test_trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
    print("Final test-set metrics:", test_metrics)

    test_novel_ratio = average_novel_ngram_ratio(model, tokenizer, test_dataset, sample_size=30)
    print(f"Final test-set novel 3-gram ratio: {test_novel_ratio:.4f} (higher is better for B)")

## 6. A/B contrast — the check that actually matters

Runs B and A on the same held-out chunks (same article `05` used) for a direct comparison.


In [ ]:
if not DATA_READY:
    print("No data — skipping.")
elif not API_KEY_SET:
    print("No NVIDIA_NIM_API_KEY (chunking needs the embedding API) — skipping the A/B contrast.")
else:
    import yaml
    from datasets import load_dataset

    from src.pipeline.chuncking import paginate_semantic, plain_text_to_paragraphs
    from src.pipeline.embeddings import embed_texts, load_config

    test_article = load_dataset("cnn_dailymail", "3.0.0", split="test[0:1]")[0]["article"]
    chunk_cfg = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))
    embed_cfg = load_config("configs/importance_filter.yaml")
    chunks = paginate_semantic(
        plain_text_to_paragraphs(test_article),
        min_words=chunk_cfg["min_words"], max_words=chunk_cfg["max_words"],
        granularity="paragraph", config=embed_cfg, embed_fn=embed_texts,
    )[:5]
    print(f"chunks: {len(chunks)}\n")

    device = next(model.parameters()).device
    rows = []
    for chunk in chunks:
        ids = tokenizer(chunk.text, return_tensors="pt", truncation=True, max_length=512).input_ids.to(device)
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=64)
        elaborated = tokenizer.decode(out[0], skip_special_tokens=True)
        rows.append(
            {
                "chunk": chunk.index,
                "source_chars": len(chunk.text),
                "output_chars": len(elaborated),
                "compression": round(len(elaborated) / max(1, len(chunk.text)), 3),
                "novel_3gram": round(novel_ngram_ratio(elaborated, chunk.text, n=3), 4),
                "text": elaborated[:110],
            }
        )

    import pandas as pd

    contrast = pd.DataFrame(rows)
    print(contrast.to_string(index=False))
    print(f"\nB mean novel 3-gram ratio: {contrast.novel_3gram.mean():.4f}")
    print("A (05, same article) stays near zero and only from sentence seams.")
    print("A clear gap here is the manipulation check passing.")

## Summary

_To be filled in after the run._

1. **Novel 3-gram ratio vs source** (§5-6): B must sit clearly above A's 0.0000.
2. **Compression ratio** (§6): XSum targets are one sentence — far harder than A's 22-32%.
3. **test ROUGE-L** (§5), not validation.

Stage 2 (rolling curation, needs QG from `09`) spec'd in `03_실험 설계`.
